# 06 pycavi SWarp

This notebooks run SWarp over the images for which Scamp creates a .head file. <br>
Tested with SWarp 2.38.0.

Uses the default configuration for SWarp, in case of needs, additional configuration parameters can be changed to optimize the performance in Chapman for example

In [ ]:
import os
import sys
import glob
import logging
from astropy.io import fits

sys.path.append(os.path.join(os.path.abspath(''), '..')) # to be removed in production

from pycavi import pycavi_tools
from pycavi.pycavi_astromatic_tools import get_swarp_lists_dict, prepare_to_swarp, run_swarp

In [ ]:
# set the directory folders
config_dir = os.getcwd() + '/' + '../config/'

config = pycavi_tools.read_pycavi_config(config_dir)
project_dir = config['Directories']['project_dir']
reduced_std_dir = config['Directories']['reduced_std_dir']
reduced_science_dir = config['Directories']['reduced_science_dir']  
swarp_dir_name = config['Directories']['swarp_dir_name'] 
log_dir = config['Directories']['project_logs_dir']

log_file_swarp = log_dir + "/pymeffpp_swarp.log"
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s:%(levelname)s:%(message)s",
                    handlers=[
                              logging.FileHandler(log_file_swarp),
                              logging.StreamHandler()])
logging.getLogger().setLevel(logging.INFO)

# all this part is to select all the .head files and grouping accordling to observation and filter
# the grouped images are stored in the swarp_list dictionary, where the key correspond to the combination
# of observation plus filter, and the element to corresponding list of images
# print(glob.glob(reduced_science_dir+"/*/*/*/*_pre/*_wcs.head"))
filters = set(['BB_B_99_ESO842', 'BB_V_89_ESO843', 'g_SDSS', 'r_SDSS', 'NB_659'])

# We clean the list of bad observations - this should be done by hand
reject_observations=[]


# We first swarp the standard stars fields
swarp_lists_std = get_swarp_lists_dict(reduced_std_dir, reject_observations, filters)
std_observation_tags = list(set(swarp_lists_std.keys()))
for i in range(len(std_observation_tags)):
    logging.info("index: {} observation_tag: {}".format(i, std_observation_tags[i]))

# We next swarp the science observation fields
swarp_lists_sci = get_swarp_lists_dict(reduced_science_dir, reject_observations, filters)
sci_observation_tags = list(set(swarp_lists_sci.keys()))    
for i in range(len(sci_observation_tags)):
    logging.info("index: {} observation_tag: {}".format(i, sci_observation_tags[i]))
    
logging.info('-------------- DONE ---------------') 

In [ ]:
# We first process the STD star observations
#
# updates some SWarp parameters 
# ---------- General processing parameters -----------------------
special_pars = {}
special_pars["RESAMPLE_DIR"] = "./resamp"
special_pars["VMEM_DIR"] = "./swap"
special_pars["COPY_KEYWORDS"] = "OBJECT,RA,DEC,OBJECT,PIXSCALE,ROT-RTA,INSTRUME,FILTER,TELESCOPE"

# ----------- Parameters governing sky subtraction ----------------
special_pars["BACK_SIZE"] = 128            # Background mesh size (pixels) all or for each image [128]
special_pars["BACK_FILTERSIZE"] = 3        # Background map filter range (meshes) (all or for each image) [3]
special_pars["BACK_FILTTHRESH"] = 0.0      # Threshold above which the background-map filter operates

# ------------Parameters governing Co-addition --------------------------

special_pars["COMBINE"] = "Y"              # Combine resampled images (Y/N)?
special_pars["COMBINE_TYPE"] = "MEDIAN"    # "MEDIAN,AVERAGE,MIN,MAX,WEIGHTED,CHI2 or SUM

# We first swarp the standard stars fields of scamp files

# If only a few data partitions write the indices to them below.
# The loop below will skip every index but those listed.
# If the list idx_selected is empty, then all data partitions will be swarped.
#
idx_selected = []

for i in range(len(std_observation_tags)):

    # skip indices not in list idx_selected
    if not (i in idx_selected or not idx_selected):
        continue
        
    observation_tag = std_observation_tags[i]
    logging.info("SWarping observation {}, filter {}".format(observation_tag.split("/")[0],
                                                             observation_tag.split("/")[1]))
    #print(swarp_lists[observation_tag])

    observation = swarp_lists_std[observation_tag][0].split("/")[-5]
    filter_ = swarp_lists_std[observation_tag][0].split("/")[-4]
    frames_list_name = observation + "_" + filter_ + ".list"
    special_pars["IMAGEOUT_NAME"] = frames_list_name[:-5] + ".coadd.fits"
    special_pars["WEIGHTOUT_NAME"] = frames_list_name[:-5] + ".weight.fits"    
    if "NB_659" in filter_:
        special_pars["SUBTRACT_BACK"] = "N"
    else:
        special_pars["SUBTRACT_BACK"] = "Y"
    swarp_dir_path = reduced_std_dir + observation + '/' + swarp_dir_name
    frames_list_name = prepare_to_swarp(swarp_lists_std[observation_tag], swarp_dir_path, frames_list_name, special_pars)
    if frames_list_name is not None:
        run_swarp(swarp_dir_path, frames_list_name, special_pars=special_pars)
    else:
        logging.info("Could not run swarp: frame_list_name is None")
    
logging.info('-------------- STD SWARP DONE ---------------') 

In [ ]:
# We first process the STD star observations
#
# updates some SWarp parameters 
# ---------- General processing parameters -----------------------
special_pars = {}
special_pars["RESAMPLE_DIR"] = "./resamp"
special_pars["VMEM_DIR"] = "./swap"
special_pars["COPY_KEYWORDS"] = "OBJECT,RA,DEC,OBJECT,PIXSCALE,ROT-RTA,INSTRUME,FILTER,TELESCOPE"

# ----------- Parameters governing sky subtraction ----------------
special_pars["BACK_SIZE"] = 128            # Background mesh size (pixels) all or for each image [128]
special_pars["BACK_FILTERSIZE"] = 3        # Background map filter range (meshes) (all or for each image) [3]
special_pars["BACK_FILTTHRESH"] = 0.0      # Threshold above which the background-map filter operates

# ------------Parameters governing Co-addition --------------------------

special_pars["COMBINE"] = "Y"              # Combine resampled images (Y/N)?
special_pars["COMBINE_TYPE"] = "MEDIAN"    # "MEDIAN,AVERAGE,MIN,MAX,WEIGHTED,CHI2 or SUM

# We first swarp the standard stars fields of scamp files

# If only a few data partitions write the indices to them below.
# The loop below will skip every index but those listed.
# If the list idx_selected is empty, then all data partitions will be swarped.
#
idx_selected = []
    
# Then we swarp the science lists of scamp files
for i in range(len(sci_observation_tags)):

    # skip indices not in list idx_selected
    if not (i in idx_selected or not idx_selected):
        continue
        
    observation_tag = sci_observation_tags[i]
    logging.info("SWarping observation {}, filter {}".format(observation_tag.split("/")[0],
                                                             observation_tag.split("/")[1]))
    #print(swarp_lists[observation_tag])

    observation = swarp_lists_sci[observation_tag][0].split("/")[-5]
    filter_ = swarp_lists_sci[observation_tag][0].split("/")[-4]
    frames_list_name = observation + "_" + filter_ + ".list"
    
    special_pars["IMAGEOUT_NAME"] = frames_list_name[:-5] + ".coadd.fits"
    special_pars["WEIGHTOUT_NAME"] = frames_list_name[:-5] + ".weight.fits"    
    if "NB_659" in filter_:
        special_pars["SUBTRACT_BACK"] = "N"
    else:
        special_pars["SUBTRACT_BACK"] = "Y"
    swarp_dir_path = reduced_science_dir + observation + '/' + swarp_dir_name
    frames_list_name = prepare_to_swarp(swarp_lists_sci[observation_tag], swarp_dir_path, frames_list_name, special_pars)
    if frames_list_name is not None:
        run_swarp(swarp_dir_path, frames_list_name, special_pars=special_pars)
    else:
        logging.info("Could not run swarp: frame_list_name is None")    
    
logging.info('-------------- SCI SWARP DONE ---------------') 